# Lab A: Chunking, Embeddings, and Saved Artifacts

> **How long:** about 50 minutes  
> **Session shape:** the morning slot explains the ideas; this notebook is the afternoon build-and-verify lab.  
> **Goal:** move the chunking and embedding logic into `smartLearn-AI/smartlearn-backend/services/rag.py`, then verify it on `pdf1.pdf`.  
> **Checkpoint:** one readable `pages.json`, one active `chunks.json`, one `.npy` embedding file, and one manifest file appear under `Day3/artifacts/`.

> **CPU-first:** everything in the main path should run on CPU. If a CUDA GPU is available, the same embedding code may run faster without changing the notebook flow.


## Before You Run

Copy the **Day3** folder provided by the TA into the same parent directory as **smartLearn-AI**.

This notebook is the afternoon implementation lab. The morning lecture already cover why RAG, chunking, and embeddings matter.

Keep two windows:
- one terminal in `smartLearn-AI`
- one notebook opened from `Day3` or its parent folder

**Windows / cmd**

```bash
cd smartLearn-AI
.\.venv\Scripts\activate.bat
```

**macOS / Linux**

```bash
cd smartLearn-AI
source .venv/bin/activate
```

Then, for both operating systems, add the Day 3 packages to `smartlearn-backend/requirements.txt` and run

```bash
python -m pip install --upgrade pip
pip install ipykernel sentence-transformers faiss-cpu pandas
```

> Optional GPU-fast path if your workstation has a CUDA-compatible GPU:
> 
> ```bash
> pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
> pip install ipykernel sentence-transformers faiss-cpu pandas
> ```


> **How to use this notebook - Balanced Vibe Coding Mode**
>
> Claude Code is a coding partner, not the tool for every keystroke. Use this decision rule first:
>
> - **Do directly:** open files, run commands, compare chunk previews, inspect saved artifacts, and read `git diff`.
> - **Use Claude Code:** add or revise behavior inside `smartlearn-backend/services/rag.py`, explain unfamiliar code, or diagnose why a chunking result looks wrong.
>
> Use the same loop each time:
>
> 1. state the visible behavior and non-goals;
> 2. ask Claude Code to inspect the relevant files and propose a short plan before editing;
> 3. approve one focused edit in `rag.py`;
> 4. inspect the real diff;
> 5. rerun the verification cell yourself.
>
> If `rag.py` is brand-new and `git diff -- smartlearn-backend/services/rag.py` shows nothing, run `git add -N smartlearn-backend/services/rag.py` once and then check the diff again.
>
> Passing behavior matters more than Claude Code saying `done`. This notebook uses `rag.py` as the single implementation target for the main path.


## 1.1 Define "Done" Before Editing

### Purpose

Write these acceptance criteria before editing:

1. `smartlearn-backend/services/rag.py` contains reusable helpers for text cleaning, page loading, chunking, embeddings, and artifact saving.
2. `paragraph`, `character`, and `character_overlap` can be compared on the same PDF.
3. The notebook can call `rag.py` and save outputs into `Day3/artifacts/`.
4. No other project file needs to change in this section.


### Locate the Day 2 Handoff Point

Day 2 already has three useful anchors:
- `smartlearn-backend/services/pdf.py` extracts page text for the small-PDF path;
- `smartlearn-backend/services/llm.py` sends the full document to the LLM;
- `smartlearn-backend/main.py` keeps the visible `/upload?chat_id=` and `/chat` contracts.


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: inspect before editing

Send Claude Code:

```text
Inspect these files and explain them in beginner language:
- smartlearn-backend/main.py
- smartlearn-backend/services/pdf.py
- smartlearn-backend/services/llm.py
- smartlearn-backend/services/rag.py

Answer four questions:
1. Which function currently reads page text?
2. Which function currently sends the full document to the LLM?
3. Which route stores the uploaded document in memory?
4. Why is rag.py the safest place to add Day 3 preparation logic first?

Do not edit any file yet.
```

After the explanation, sketch this data flow on paper:

`PDF -> pages -> chunks -> embeddings -> saved files`

**Completion evidence:** you can point to the current Day 2 boundary and say where Day 3 logic should live first.


> **Checkpoint 2 - The handoff is visible:** you can identify the existing Day 2 page loader, the full-document answer path, and the new place where reusable RAG helpers should live.


## 1.2 Start the Verification Harness

Create `rag.py` in the `smartlearn-backend/services` directory.

In [ ]:
touch smartlearn-backend/services/rag.py

### Verification

In [ ]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd()
print(cwd)
repo_name_candidates = ["smartLearn-AI"]
path_pairs = []
for repo_name in repo_name_candidates:
    path_pairs.extend([
        (cwd, cwd.parent / repo_name),
        (cwd / "Day3", cwd / repo_name),
        (cwd.parent / "Day3", cwd),
    ])
print(path_pairs)
for day3_dir, repo_dir in path_pairs:
    backend_dir = repo_dir / "smartlearn-backend"
    if (day3_dir / "pdf1.pdf").exists() and (backend_dir / "services" / "rag.py").exists():
        DAY3_DIR = day3_dir.resolve()
        REPO_DIR = repo_dir.resolve()
        BACKEND_DIR = backend_dir.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not locate Day3/pdf1.pdf and smartLearn-AI/smartlearn-backend/services/rag.py from the current working directory."
    )

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from services import rag as rag_module

rag = importlib.reload(rag_module)
ARTIFACT_ROOT = DAY3_DIR / "artifacts"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("Day3 folder:", DAY3_DIR)
print("Repository folder:", REPO_DIR)
print("Backend folder:", BACKEND_DIR)
print("Artifact root:", ARTIFACT_ROOT)


> **Expected output:** the notebook prints the Day 3 folder, the SmartLearn Lite repository folder, the artifact root, and the embedding device.


## 1.2.1 Optional Local MiniLM Model Download

### Purpose

The embedding step uses `sentence-transformers/all-MiniLM-L6-v2`. The first load usually tries Hugging Face, but it is highly recommended to download the same MiniLM model from ModelScope.

Model page: [AI-ModelScope/all-MiniLM-L6-v2](https://www.modelscope.cn/models/AI-ModelScope/all-MiniLM-L6-v2)

MiniLM maps sentences or paragraphs into 384-dimensional vectors. It can run on CPU; a CUDA GPU only makes the embedding step faster.

Recommended target folders:
- for this notebook: `Day3/artifacts/hf_models/all-MiniLM-L6-v2`
- for the web backend upload route: `smartLearn-AI/smartlearn-backend/artifacts/rag/hf_models/all-MiniLM-L6-v2`

From the `smartLearn-AI` folder, use cmd:

```bash
cd smartLearn-AI
./.venv/Scripts/activate.bat
python -m pip install "modelscope==1.27.1"
modelscope download --model AI-ModelScope/all-MiniLM-L6-v2 --local_dir ../Day3/artifacts/hf_models/all-MiniLM-L6-v2
modelscope download --model AI-ModelScope/all-MiniLM-L6-v2 --local_dir smartlearn-backend/artifacts/rag/hf_models/all-MiniLM-L6-v2
```

`modelscope` is only a download helper here. The app still loads the finished local model folder through `sentence-transformers`.


In [ ]:
MODEL_LOCAL_NAME = "all-MiniLM-L6-v2"
MODEL_REQUIRED_FILES = [
    "modules.json",
    "config_sentence_transformers.json",
    "1_Pooling/config.json",
]

model_targets = {
    "notebook": ARTIFACT_ROOT / "hf_models" / MODEL_LOCAL_NAME,
    "backend": BACKEND_DIR / "artifacts" / "rag" / "hf_models" / MODEL_LOCAL_NAME,
}

def local_model_ready(path: Path) -> bool:
    return all((path / item).exists() for item in MODEL_REQUIRED_FILES)

for label, path in model_targets.items():
    status = "ready" if local_model_ready(path) else "missing"
    print(f"{label}: {status} -> {path}")

candidate = next((path for path in model_targets.values() if local_model_ready(path)), None)
if candidate is None:
    print("Run the ModelScope download command above, then rerun this cell.")
else:
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(
            str(candidate),
            device="cpu",
            model_kwargs={"use_safetensors": False},
        )
        vectors = model.encode(
            ["MiniLM can create local CPU embeddings."],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        print("Loaded local model:", candidate)
        print("Embedding shape:", vectors.shape)
    except Exception as exc:
        print("Local model folder was found, but loading failed:", repr(exc))


> **Expected output:** the notebook model path should be `ready` before Section 1.7 runs. If the backend model path is also `ready`, the frontend upload route can build embeddings without fetching MiniLM from Hugging Face.


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: move the page-loading helpers into `rag.py`

Send Claude Code:

```text
Inspect smartlearn-backend/services/rag.py and smartlearn-backend/services/pdf.py.

In smartlearn-backend/services/rag.py, add or revise these helpers:
- `clean_text`: normalize one extracted page of PDF text by removing null bytes, soft hyphens, repeated whitespace, and noisy line breaks
- `extract_pages_for_rag`: read the PDF page by page, keep the original PDF page numbers, and return only readable `{page, text}` records
- `save_json`: save one Python object to a UTF-8 JSON file and create parent folders when needed
- `load_json`: read one saved JSON artifact back into Python
- `preview_records`: show a small notebook table for chosen columns so we can inspect page/chunk artifacts quickly

Requirements:
- keep original PDF page numbers
- remove empty extracted text blocks
- do not hard-code a 30-page limit here
- accept either str or Path for file paths when reasonable
- keep the return shape as a list of {page, text}
- do not edit other project files in this step

Before editing, state which functions already exist and what will change.
```

### Review map after the edit

Open the diff and confirm:
- `extract_pages_for_rag(...)` is separate from the Day 2 small-PDF route helper;
- page numbers still match the original PDF;
- no absolute local path is hard-coded;
- empty pages do not become fake chunks later.

**Completion evidence:** `rag.extract_pages_for_rag(...)` reads `pdf1.pdf` and returns a readable preview table.


## 1.3 Vibe Code Text Cleaning and Page Loading

### Purpose

This is the Day 2 prerequisite in reusable form. The output must still be `[{page, text}]`, but the helper should now live in `rag.py` and work for longer PDFs too.


A PDF is still a document, but the code can only work with text records. In practice, this step extracts page text, applies light cleaning, and keeps one record per page for later chunking.


In [ ]:
from services import rag as rag_module
rag = importlib.reload(rag_module)
def preview_records(records: list[dict], columns: list[str], rows: int = 5):
    try:
        import pandas as pd
    except ImportError as exc:
        raise ImportError("pandas is required for preview_records") from exc

    frame = pd.DataFrame(records)
    if frame.empty:
        return frame
    usable_columns = [column for column in columns if column in frame.columns]
    return frame[usable_columns].head(rows)


pd.set_option("display.max_colwidth", 150)
pdf1_pages = rag.extract_pages_for_rag(DAY3_DIR / "pdf1.pdf")
print(f"Extracted pages from pdf1.pdf: {len(pdf1_pages)}")
preview_records(pdf1_pages, columns=["page", "text"], rows=10)


: 

> **Checkpoint 3 - The page loader works independently:** `pdf1.pdf` becomes `[{page, text}]`, page numbers are preserved, and the preview shows real extracted text.


## 1.5 Vibe Code the Three Chunking Modes

### Purpose

Long documents must be split into smaller pieces that an embedding model and retriever can handle. Different chunking rules keep different amounts of context, so retrieval quality can change. In this section, keep three simple modes side by side so their trade-offs stay visible.

- `paragraph`
- `character`
- `character_overlap`


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: implement chunking as three clear options

Send Claude Code:

```text
Inspect `smartlearn-backend/services/rag.py`.

Before editing, explain in one sentence how each chunking mode should differ.

Add or revise the chunking helpers so that `rag.py` supports:

- `slice_long_text`: Split a single oversized text block into smaller pieces, preferring natural boundaries and avoiding splits in the middle of words whenever possible.
- `chunk_by_paragraph`: Convert paragraph-level records into chunks while preserving page numbers and paragraph order.
- `chunk_by_characters`: Create plain fixed-size sliding-window chunks, with optional overlap metadata.
- `build_chunks(records, mode, chunk_mode, chunk_size, overlap)`: Select the requested chunking strategy and return a uniform chunk schema for the rest of the pipeline.

`chunk_mode` must support exactly these three values:

- `"paragraph"`
- `"character"`
- `"character_overlap"`

Requirements:

- Paragraph mode should preserve paragraph boundaries as much as possible.
- When a single paragraph exceeds `chunk_size`, split it into smaller pieces without starting a new piece in the middle of a word whenever possible.
- Character mode must use plain fixed-size windows with no overlap.
- Character-overlap mode must reuse the character chunking logic with `overlap > 0`.
- Every chunk must contain at least `chunk_id`, `page`, `text`, and `chunk_mode`.
- Preserve the original page number and text order.
- Do not edit any other project files.
```

### Review map after the edit

When you inspect the diff, locate:
- where the page number is copied into each chunk;
- where overlap changes the next start position;
- where paragraph mode decides to flush or split a long block.

**Completion evidence:** one comparison table shows chunk count, average length, and the boundary preview for all three modes.


In [ ]:
from services import rag as rag_module
rag = importlib.reload(rag_module)
chunk_compare_rows = []
for mode in ["paragraph", "character", "character_overlap"]:
    chunks = rag.build_chunks(
        pdf1_pages,
        chunk_mode=mode,
        chunk_size=700,
        overlap=120,
    )
    avg_chars = int(np.mean([len(item["text"]) for item in chunks]))
    chunk_compare_rows.append(
        {
            "chunk_mode": mode,
            "num_chunks": len(chunks),
            "avg_chars": avg_chars,
            "sample_page": chunks[0]["page"],
            "chunk_1_preview": chunks[0]["text"][:100],
            "chunk_1_tail": chunks[0]["text"][-80:],
            "chunk_2_head": chunks[1]["text"][:80] if len(chunks) > 1 else "",
        }
    )

chunk_compare_df = pd.DataFrame(chunk_compare_rows)
print(chunk_compare_df[["chunk_mode", "num_chunks", "avg_chars"]])
chunk_compare_df


> **Checkpoint 4 - Chunking trade-offs are visible:** you can point to one example where fixed-size chunks cut through a sentence and one example where paragraph mode keeps a cleaner boundary.

> Try out Appendix A for better paragraph chunking!


## 1.6 Pick One Active Storage Configuration

### Purpose

The notebook compares three modes, but the saved artifacts should come from one active setting so Section 2 can reuse them directly.

Use this default active setup first:
- `ACTIVE_CHUNK_MODE = "character_overlap"`
- `CHUNK_SIZE = 700`
- `OVERLAP = 120`
- `EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"`


In [ ]:
ACTIVE_CHUNK_MODE = "character_overlap"
CHUNK_SIZE = 700
OVERLAP = 120
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

active_chunks = rag.build_chunks(
    pdf1_pages,
    chunk_mode=ACTIVE_CHUNK_MODE,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
)

pd.set_option("display.max_colwidth", 100)
print("Active mode:", ACTIVE_CHUNK_MODE)
print("Chunk count:", len(active_chunks))
preview_records(active_chunks, columns=["chunk_id", "page", "chunk_mode", "text"], rows=10)


## 1.7 Vibe Code the Embedding Pipeline

### Purpose

An embedding model turns each chunk into a numeric vector that captures rough semantic meaning. Later, retrieval compares the question vector with these chunk vectors to find likely evidence.


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: generate embeddings and save reusable files

Send Claude Code:

```text
Inspect smartlearn-backend/services/rag.py.

Add or revise the helpers for the embedding pipeline:
- `model_tag`: turn a model name into a safe filename suffix for saved artifacts
- `resolve_model_source`: prefer a local cached model folder when it already exists
- `get_device`: choose CPU or CUDA for the current machine
- `load_model`: create or reuse one sentence-transformer model instance
- `embed_texts`: encode a list of texts into normalized `float32` vectors
- `artifact_paths_for`: decide where pages, chunks, embeddings, manifests, and indexes should be saved
- `ensure_artifacts(document_id, pdf_name, pages, chunk_mode, model_name, chunk_size, overlap, batch_size, artifact_root)`: build or reuse the full pages -> chunks -> embeddings -> manifest bundle for one PDF and one active config

Requirements:
- CPU-first; if CUDA is available, the same code may use it automatically
- support an optional artifact_root argument so the notebook can save into Day3/artifacts
- save raw pages, chunk metadata, embeddings, and a manifest
- reuse saved outputs when the signature still matches
- record document_id, pdf_name, num_pages, chunk_mode, chunk_size, overlap, model_name, num_chunks, embedding_dim, device, chunk_path, embedding_path, raw_pages_path in the manifest
- do not edit other project files

Before editing, list the files that should appear after one successful run.
```

### Review map after the edit

Check the diff for these ideas:
- model loading is separate from embedding generation;
- the cache path and artifact path are not confused;
- the manifest is the place that records the active configuration;
- the notebook can override the artifact root without changing the project default.

**Completion evidence:** one call to `rag.ensure_artifacts(...)` creates reusable files under `Day3/artifacts/`.


In [ ]:
from services import rag as rag_module
rag = importlib.reload(rag_module)
pdf1_bundle = rag.ensure_artifacts(
    document_id="pdf1",
    pdf_name="pdf1.pdf",
    pages=pdf1_pages,
    chunk_mode=ACTIVE_CHUNK_MODE,
    model_name=EMBED_MODEL_NAME,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    batch_size=32,
    artifact_root=ARTIFACT_ROOT,
)

{
    "num_pages": pdf1_bundle["manifest"]["num_pages"],
    "num_chunks": pdf1_bundle["manifest"]["num_chunks"],
    "embedding_dim": pdf1_bundle["manifest"]["embedding_dim"],
    "device": pdf1_bundle["manifest"]["device"],
    "chunk_path": pdf1_bundle["manifest"]["chunk_path"],
    "embedding_path": pdf1_bundle["manifest"]["embedding_path"],
}


In [ ]:
print("Chunk count:", len(pdf1_bundle["chunks"]))
print("Embedding shape:", pdf1_bundle["embeddings"].shape)
print("Device used:", pdf1_bundle["manifest"]["device"])
assert len(pdf1_bundle["chunks"]) == pdf1_bundle["embeddings"].shape[0]


## 1.8 Commit the Chunking and Embedding Milestone

### Purpose

This section turned PDF pages into reusable chunk files, embedding files, and a manifest. Save this working state before retrieval logic makes the pipeline larger.

### Step 1: Confirm the section evidence

Before Git commands, confirm:

- `pages.json` exists under `Day3/artifacts/pdf1/`;
- one chunk file exists under `Day3/artifacts/pdf1/`;
- one embedding `.npy` file and one manifest exist under `Day3/artifacts/pdf1/`;
- the notebook printed chunk count, embedding shape, and device;
- `smartlearn-backend/services/rag.py` now contains the reusable helpers.

If one item is missing, fix that first.

### Step 2: Inspect the working tree

From the `smartLearn-AI` repository root, run:

```bash
git status
```

These files should not be staged:

- `Day3/artifacts/` ? generated outputs;
- `.venv/` ? local environment;
- sentence-transformer cache folders ? downloaded model files;
- `__pycache__/` and `*.pyc` ? generated cache;
- `.env` ? local secrets.

### Step 3: Review and stage only the reusable pipeline code

Run:

```bash
git diff -- smartlearn-backend/services/rag.py smartlearn-backend/requirements.txt
git add smartlearn-backend/services/rag.py smartlearn-backend/requirements.txt
git status
git diff --staged
```

Confirm the staged diff shows:

- page loading and chunk building helpers;
- embedding and artifact helpers;
- dependency updates only when you actually added a new package.

### Step 4: Create and verify the commit

Run:

```bash
git commit -m "feat: add RAG chunking and embedding pipeline"
git log -1 --oneline
git status
```

> **Expected output:** One commit records the reusable chunking + embedding pipeline, and generated artifacts stay untracked.


In [ ]:
git status
git diff -- smartlearn-backend/services/rag.py smartlearn-backend/requirements.txt
git add smartlearn-backend/services/rag.py smartlearn-backend/requirements.txt
git status
git diff --staged
git commit -m "feat: add RAG chunking and embedding pipeline"
git log -1 --oneline
git status


> ✅ **Checkpoint 8 — The chunking and embedding milestone is committed:** The repo contains the reusable `rag.py` pipeline and `requirements.txt` updates, while generated artifacts remain outside the commit.
>
> **If it does not match:** Unstage unrelated files, re-check the saved artifact paths, and commit only after you can explain every staged line.


## Section 1 Checkpoint

- [ ] I can explain why the Day 2 full-document prompt is not enough for longer PDFs.
- [ ] I can read `pdf1.pdf` into `[{page, text}]`.
- [ ] I compared `paragraph`, `character`, and `character_overlap`.
- [ ] I saved raw pages, chunks, embeddings, and a manifest under `artifacts/`.
- [ ] The code lives in `smartlearn-backend/services/rag.py`, not in notebook-only helper cells.

**Expected output**
- `artifacts/raw_pages/pdf1_pages.json`
- `artifacts/chunks/pdf1_character_overlap.json`
- `artifacts/embeddings/pdf1_character_overlap_all_MiniLM_L6_v2.npy`
- `artifacts/embeddings/pdf1_character_overlap_all_MiniLM_L6_v2.manifest.json`

**Homework for Day 4**
1. Change `pdf_name` to the larger `pdf2.pdf` and record how long the embedding step takes on your machine.
2. Try `paragraph` mode once and write one good case for it.
3. Bring one retrieval question from `pdf1.pdf` to Section 2.


## Appendix A: LangChain `RecursiveCharacterTextSplitter`

This appendix keeps the same idea as Sections 1.5 and 1.6, but uses a library splitter that tries larger separators first and smaller separators later.

Common separator order:
- `"\n\n"`
- `"\n"`
- `" "`
- `""`

Install first if needed:

```bash
pip install langchain-text-splitters
```

Don't forget to add it in `smartlearn-backend/requirements.txt` if your workshop branch will keep using it.


<!-- BEGINNER-WALKTHROUGH -->

### Vibe Coding walkthrough: implement a cleaner splitter with LangChain, then test it here

Send Claude Code:

```text
Inspect smartlearn-backend/services/rag.py.

Add one optional LangChain-based chunking helper for messy PDF text.
The goal is to split each page more cleanly than the plain paragraph helper when paragraph boundaries are noisy or incomplete.

Implementation requirements:
- keep the existing paragraph, character, and character-overlap paths unchanged
- add a helper named `chunk_with_langchain_recursive(pages, chunk_size, chunk_overlap, separators=None)`
- use RecursiveCharacterTextSplitter with a separator priority like: double newline, single newline, space, then character-level fallback
- keep every returned chunk in the same record format used elsewhere in rag.py, including page number, text, chunk id, and a clear chunk mode
- skip empty chunks and preserve chunk-to-page mapping
- if `langchain-text-splitters` is needed, add it to `smartLearn-AI/smartlearn-backend/requirements.txt`
- if langchain-text-splitters is not installed yet, raise a clear ImportError that tells the user what to install
- make sure the next notebook test cell can call `rag.chunk_with_langchain_recursive(...)` directly for side-by-side comparison with the original paragraph split

If it fits the current file structure cleanly, also extend `build_chunks(pages, chunk_mode, chunk_size, overlap)` so `chunk_mode="langchain_recursive"` calls `chunk_with_langchain_recursive(...)` without changing the default Section 1 path.

After editing, explain:
- how the recursive splitter chooses boundaries
- why its chunk starts may look cleaner on PDF text
- that the next notebook cell should run `rag.chunk_with_langchain_recursive(...)` first, and the comparison cell after that should still use `rag.build_chunks(..., chunk_mode="paragraph", ...)`
```

**Completion evidence:** the next notebook cell can call the new helper directly and show a side-by-side comparison with the original paragraph split.


In [ ]:
try:
    langchain_recursive_chunks = rag.chunk_with_langchain_recursive(
        pdf1_pages,
        chunk_size=700,
        chunk_overlap=120,
    )
    print(f"LangChain recursive chunk count: {len(langchain_recursive_chunks)}")
    preview_records(
        langchain_recursive_chunks,
        columns=["chunk_id", "page", "chunk_mode", "text"],
        rows=5,
    )
except ImportError as exc:
    print(exc)
    print("langchain-text-splitters should be installed first.")


In [ ]:
if "langchain_recursive_chunks" not in globals():
    print("No LangChain comparison yet.")
    compare_df = None
else:
    paragraph_chunks = rag.build_chunks(
        pdf1_pages,
        chunk_mode="paragraph",
        chunk_size=700,
        overlap=120,
    )

    compare_rows = []
    for mode_name, chunks in [
        ("paragraph_original", paragraph_chunks),
        ("langchain_recursive", langchain_recursive_chunks),
    ]:
        avg_chars = int(np.mean([len(item["text"]) for item in chunks]))
        compare_rows.append(
            {
                "chunk_mode": mode_name,
                "num_chunks": len(chunks),
                "avg_chars": avg_chars,
                "sample_page": chunks[0]["page"],
                "chunk_1_preview": chunks[0]["text"][:100],
                "chunk_1_tail": chunks[0]["text"][-80:],
                "chunk_2_head": chunks[1]["text"][:80] if len(chunks) > 1 else "",
            }
        )

    compare_df = pd.DataFrame(compare_rows)
compare_df


### Appendix A Checkpoint

- [ ] I can explain what separator priority means.
- [ ] I can compare the first boundary of the LangChain split with the plain paragraph split.
- [ ] I know this appendix is optional and does not replace the main Section 1 milestone.
